## Exploring Experiment Results
This notebook allows for exploration of experiment results.

In [18]:
import pandas as pd
from sklearn.metrics import classification_report

In [ ]:
# Experiment we want to explore - customise these
TARGET_LABEL = "object_region"
EXPERIMENT_NUMBER = 4
# Use 0 to display every row, or a 1-based row number within the experiment.
ROW_NUMBER = 1

In [20]:
# Generated constants
FOLDER_NAME = f"{TARGET_LABEL}_classify"
RESULTS_PATH = f"../results/{FOLDER_NAME}"
EXPERIMENTS_DF_PATH = f"{RESULTS_PATH}/experiments.parquet"

In [21]:
def get_experiment_rows(df, experiment_number, row_number=0):
    """
    Return the requested 1-based row(s) within an experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Returns:
        pd.DataFrame: The selected row as a one-row DataFrame, or every row for the
            experiment when row_number is 0.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    # Filter the DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]

    # Catch errors for invalid experiment numbers or row numbers
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")
    if row_number < 0:
        raise ValueError("ROW_NUMBER must be 0 or a positive integer.")
    if row_number > len(df_experiment):
        raise ValueError(
            f"Row {row_number} not found in experiment {experiment_number}; "
            f"it contains {len(df_experiment)} row(s)."
        )
    # Return all rows if row_number is 0 or one row if a valid row_number is provided
    if row_number == 0:
        return df_experiment
    return df_experiment.iloc[[row_number - 1]]

In [22]:
def print_experiment_info(df, experiment_number, row_number=0):
    """
    Print the model and data configurations, training history, and evaluation results for a
    given experiment number and optional 1-based row number. A row number of 0
    prints every row.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the information for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        # Print model and data configurations
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")
        print("Train Config:")
        for key, value in row["train_config"].items():
            print(f"  {key}: {value}")
        print("\nData Config:")
        for key, value in row["data_config"].items():
            print(f"  {key}: {value}")

        # Print training history
        history = row["history"]
        print("Training History:")
        for epoch_idx, metrics in enumerate(history):
            print(f"  Epoch {epoch_idx + 1}:")
            for metric_name, metric_value in metrics.items():
                print(f"    {metric_name}: {metric_value}")

        # Print evaluation results
        print("\nStandard Test Results:")
        print(f"  Test Accuracy: {row['test_acc']:.4f}")
        print(f"  Test Loss: {row['test_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_weighted_f1_avg']:.4f}")

        print("\nUnseen Matched Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_matched_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_matched_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_matched_weighted_f1_avg']:.4f}")

        print("\nUnseen Related Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_related_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_related_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_related_weighted_f1_avg']:.4f}")

In [23]:
def print_classification_report(df, experiment_number, row_number=0):
    """
    Print classification reports for the standard test set. A row number of 0
    prints a report for every row in the selected experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Classification Reports for Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the classification report for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")

        # Generate and print the classification report
        report = classification_report(
            row["test_y_true"], row["test_y_pred"], target_names=row["train_labels"]
        )
        print(report)

In [24]:
# Load the required experiment results from the Parquet file
df = pd.read_parquet(EXPERIMENTS_DF_PATH)
df.tail(10)

,experiment_number,experiment_name,run_name,seed,deterministic,freeze_backbone,pretrained_checkpoint,train_config,data_config,model_type,...,test_unseen_matched_weighted_f1_avg,test_unseen_matched_y_true,test_unseen_matched_y_expected,test_unseen_matched_y_pred,test_unseen_related_acc,test_unseen_related_loss,test_unseen_related_weighted_f1_avg,test_unseen_related_y_true,test_unseen_related_y_expected,test_unseen_related_y_pred
16,4,resnet18_all_augs_bg_sweep,bg_subtraction_on,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.627836,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",26.111111,5.094202,0.345934,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 0, 11, 11, 11, 11, 11, 11..."
17,5,resnet18_pad_aug_sweep,none,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.488622,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",18.930556,5.987170,0.235327,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 0, 0, 0, 0, 11, 1..."
18,5,resnet18_pad_aug_sweep,color_jitter,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.546015,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",22.569444,5.543232,0.287506,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 0, 0, 11, 11, 11, 11, 11, 11, 11, 11,..."
19,5,resnet18_pad_aug_sweep,flip,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.499485,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 3, 12, 12, 12...",19.430556,5.938330,0.235409,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 1..."
20,5,resnet18_pad_aug_sweep,both,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.530288,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",22.027778,5.564985,0.277273,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[0, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11..."
21,6,resnet18_color_jitter_sweep,ssvtp_settings,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.514104,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",21.500000,5.554856,0.274740,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[11, 11, 11, 0, 0, 11, 11, 11, 11, 11, 11, 11,..."
22,6,resnet18_color_jitter_sweep,t3_settings,129,True,False,NaN,{'checkpoint_dir': '/content/drive/MyDrive/Col...,"{'batch_size': 32, 'bg_path': 'data/baseline.j...",resnet18,...,0.552776,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1...",28.347222,4.386106,0.369691,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 

In [25]:
print_experiment_info(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Experiment Number: 4
Experiment Name: resnet18_all_augs_bg_sweep


---- Row 1: bg_subtraction_off ----

Train Config:
  checkpoint_dir: /content/drive/MyDrive/Colab Notebooks/touch-ex/checkpoints/object_classify/004
  early_stopping_min_delta: 0.1
  early_stopping_patience: 5.0
  learning_rate: 2e-05
  min_learning_rate: 1e-06
  model_title: bg_subtraction_off
  momentum: 0.9
  num_epochs: 25
  optimizer: adamw
  warmup_epochs: 1
  warmup_start_factor: 0.1
  weight_decay: 0.02

Data Config:
  batch_size: 32
  bg_path: None
  filtered_force_level: None
  filtered_motion: None
  norm_cache_path: configs/norm_cache.json
  norm_type: dataset
  num_workers: 4
  random_state: 129
  shuffle_map: {'test': False, 'train': True, 'val': False}
  split_size: 0.2
  stratify_label: object
  train_augmentations: {'color_jitter': {'brightness': array([0.9, 1.1]), 'contrast': array([0.9, 1.1]), 'hue': 0.05, 'saturation': 0.2}, 'horizontal_flip': 0.5, 'random_resized_crop': True}
  transform_name: cente

In [26]:
print_classification_report(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Classification Reports for Experiment Number: 4
Experiment Name: resnet18_all_augs_bg_sweep


---- Row 1: bg_subtraction_off ----

                precision    recall  f1-score   support

      football       0.98      1.00      0.99       960
        hammer       0.73      0.68      0.70       840
           mug       0.79      0.91      0.85       840
plastic_bottle       0.70      0.84      0.77       840
      scissors       0.81      0.69      0.75       840
sponge_scourer       0.94      0.95      0.94       840
     tea_towel       0.90      0.95      0.92       840
   tennis_ball       0.99      0.99      0.99       960
     tin_beans       0.85      0.85      0.85       840
   toilet_roll       0.93      0.80      0.86       840
    toothbrush       0.96      0.86      0.90       840
 tube_pringles       0.89      0.78      0.83       840
     tv_remote       0.79      0.88      0.83       960
  wooden_spoon       0.79      0.83      0.81       840

      accuracy             